# DeepWeeds — Nguyen Tuan Thanh — 2A202602640

Bật **Runtime > Change runtime type > T4 GPU** rồi chạy các ô theo thứ tự.
Đưa thư mục dự án lên Google Drive; đặt `PROJECT_DIR` trỏ tới thư mục đó. Không cần ZIP đầu vào.

Profile gọn: **5 backbone × 12 epoch**, 3 trục ablation, 1 kết hợp,
baseline và chung kết đủ **3 seed**. Thời gian có thể vượt hạn 2 giờ.
Mọi kết quả là từ lần chạy riêng; chưa có kết quả train trong gói này.


In [ ]:
# Cài phần bổ sung; giữ nguyên torch/torchvision CUDA của Colab.
%pip -q install timm==1.0.29 fvcore==0.1.5.post20221221 openpyxl scikit-learn
import torch
assert torch.cuda.is_available(), "Bật T4 GPU trong Runtime > Change runtime type."
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
PROJECT_DIR = Path('/content/drive/MyDrive/K4-Track4-Day2-Deeplearning-Advance')
RUN_NAME = "epochs12_v1"  # giữ nguyên khi resume; tách khỏi các lần đã chạy 5 epoch
PROFILE = "deadline"
EPOCHS = 12               # ngân sách huấn luyện mỗi model
BATCH_SIZE = 64           # nếu OOM: bắt đầu RUN_NAME mới với 32; không đổi giữa vòng so sánh
NOTEBOOK_LINK = ""        # dán link notebook Colab của bạn trước khi xuất bài
ROOT = Path('/content/deepweeds_lab')
DATA = Path('/content/deepweeds_data')
OUTPUT = Path('/content/drive/MyDrive/DeepWeeds_2A202602640') / RUN_NAME / '2A202602640_nguyen_tuan_thanh'
print("Kết quả:", OUTPUT)


In [ ]:
# Đọc trực tiếp dự án trên Drive, chỉ chép code/tài liệu xuống /content cho nhanh.
import shutil, sys, os
assert (PROJECT_DIR / 'eval.py').is_file(), (
    f"Không thấy {PROJECT_DIR}/eval.py. Sửa PROJECT_DIR thành đường dẫn thư mục dự án trên Drive."
)
SUBMISSION = 'submissions/2A202602640_nguyen_tuan_thanh'
assert (PROJECT_DIR / SUBMISSION / 'code/colab_runner.py').is_file(), (
    "Thư mục Drive chưa có code mới. Upload lại thư mục submissions/2A202602640_nguyen_tuan_thanh."
)
ROOT.mkdir(parents=True, exist_ok=True)
for name in ('eval.py', 'README.md', 'GUIDE.md', 'RUBRIC.md'):
    shutil.copy2(PROJECT_DIR / name, ROOT / name)
for name in ('tests', 'starter'):
    shutil.copytree(PROJECT_DIR / name, ROOT / name, dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns('__pycache__', '*.pyc'))
local_sub = ROOT / SUBMISSION
local_sub.mkdir(parents=True, exist_ok=True)
shutil.copy2(PROJECT_DIR / SUBMISSION / 'README.md', local_sub / 'README.md')
shutil.copytree(PROJECT_DIR / SUBMISSION / 'code', local_sub / 'code', dirs_exist_ok=True,
                ignore=shutil.ignore_patterns('__pycache__', '*.pyc'))
CODE = local_sub / 'code'
os.environ['LAB_REPO_DIR'] = str(ROOT)
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(CODE))
# Nạp lại code khi người dùng cập nhật thư mục Drive trong cùng phiên Colab.
for name in ('colab_runner', 'experiments', 'train', 'dataset', 'model', 'losses',
             'inference', 'benchmark', 'make_results', 'bonus'):
    sys.modules.pop(name, None)
import colab_runner as runner
plan = runner.make_plan(PROFILE, EPOCHS, BATCH_SIZE)


In [ ]:
# Các kiểm tra CPU trước training (không tốn GPU để train model).
import subprocess
subprocess.run([sys.executable, str(CODE / 'selftest.py')], check=True)
subprocess.run([sys.executable, str(CODE / 'test_workflow.py')], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', str(ROOT / 'tests')], cwd=ROOT, check=True)


In [ ]:
# Tải ~490 MB, kiểm tra checksum, giải nén trên /content cho nhanh.
runner.prepare(DATA)
lab = runner.create_lab(DATA, OUTPUT, plan)
runner.run_stages(lab, ['0'])  # kiểm tra split/EDA/overfit một batch


## Vòng 1 — So sánh 5 backbone

Giữ cùng seed, split và công thức. Ô này in thời gian thật để ước lượng phần còn lại.
Các lần đã hoàn thành được bỏ qua khi resume, sau khi kiểm tra checkpoint và cấu hình.


In [ ]:
runner.run_stages(lab, ['1'])


## Vòng 2 — Công thức huấn luyện và 3 seed

Khởi tạo: fine-tune/đóng băng. Augmentation: cơ bản/CutMix. Loss: CE/label smoothing.
Thử kết hợp tốt nhất; tái sử dụng cấu hình trùng chính xác, có ghi nguồn trong log.


In [ ]:
runner.run_stages(lab, ['2'])


## Vòng 3 — Suy luận trên validation

TTA, crop, độ phân giải, gộp logit/xác suất, ensemble, temperature scaling và FP16/AMP.
Độ trễ warmup 10, đo 50 lần, có đồng bộ GPU, batch 1 và batch 32.


In [ ]:
runner.run_stages(lab, ['3'])


## Chung kết — Mở test sau khi chốt cấu hình

Chạy test một lần mỗi seed; marker và cache ngăn chạy lại khi phiên bị ngắt.
Các file chưa/sau hiệu chuẩn được tạo từ cùng lượt forward.


In [ ]:
runner.run_stages(lab, ['4'])


## Xuất bài nộp

Điền link notebook của bạn vào `NOTEBOOK_LINK`, sau đó chạy ô dưới.
Đọc lại `report.md`, xác nhận nhận xét ảnh sai đã đánh dấu và đặt quyền chia sẻ notebook.


In [ ]:
runner.run_stages(lab, ['5'], notebook_link=NOTEBOOK_LINK or None)
archive = OUTPUT.parent / 'submission_2A202602640.zip'
print("Bài nộp lưu trên Drive:", archive)
from IPython.display import Markdown, display
display(Markdown((OUTPUT / 'report.md').read_text()))


In [ ]:
# ZIP đầu ra tự tạo để tải bài nộp; bạn không cần tự nén thư mục đầu vào.
from google.colab import files
files.download(str(OUTPUT.parent / 'submission_2A202602640.zip'))


## Tiếp tục khi Colab ngắt

Mount cùng Drive, giữ nguyên PROJECT_DIR/RUN_NAME/EPOCHS/BATCH_SIZE.
Chạy lại ô tải dữ liệu, sau đó các vòng cần tiếp tục. Log và checkpoint lưu trên Drive.
Không dùng kết quả test để thay đổi cấu hình. Thay đổi cấu hình cần RUN_NAME mới.

Toàn bộ số liệu được xuất từ log và dự đoán của lần chạy này.
